# 02 · Corpus Analysis

تحليل أنماط التصيد في البيانات الأصلية لفهم كيف يبني المحتالون رسائلهم.

---


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent.parent))

import pandas as pd
import re
from collections import Counter

df = pd.read_csv('../../data/raw/sms_dataset.csv').drop_duplicates(subset='text')
phish = df[df['Label']==0]['text']
safe = df[df['Label']==1]['text']
print(f'Corpus: {len(phish)} phishing + {len(safe)} safe')


Corpus: 596 phishing + 590 safe


## تحليل الروابط


In [2]:
url_re = re.compile(r'[a-zA-Z0-9\-]+\.[a-zA-Z]{2,}')
phish_urls = phish.apply(lambda t: bool(url_re.search(str(t)))).sum()
print(f'Phishing with URLs: {phish_urls}/{len(phish)} ({phish_urls/len(phish)*100:.0f}%)')

domains = []
for t in phish:
    domains.extend(url_re.findall(str(t)))
print('\nTop 10 phishing domains:')
for dom, cnt in Counter(domains).most_common(10):
    print(f'  {dom:35s} {cnt}')


Phishing with URLs: 571/596 (96%)

Top 10 phishing domains:
  stc-portal.net                      78
  mobily-rewards.cc                   76
  zain-pay.online                     70
  bank-card-activation.online         10
  bank-secure-login.cc                10
  bank-rewards-claim.info             10
  bank-system-upgrade.top             9
  bank-update-portal.net              9
  dhl-express-secure.net              6
  spl-sa-verify.info                  5


## تحليل الكلمات المفتاحية


In [3]:
groups = {
    'تهديد': ['تحذير','تنبيه','إيقاف','حظر','تعليق','تجميد','مغلق'],
    'مكافآت': ['فزت','مبروك','تهانينا','جائزة','ربحت','مكافأة'],
    'استعجال': ['عاجل','فوراً','الآن','خلال 24','سارع'],
    'حث': ['اضغط','ادخل','قم بالدخول','يرجى الدخول'],
}
for group, kw in groups.items():
    n = phish.apply(lambda t: any(k in str(t) for k in kw)).sum()
    bar = '█' * int(n/len(phish)*100/3)
    print(f'  {group:10s} {bar} {n} ({n/len(phish)*100:.0f}%)')


  تهديد      ████████████ 226 (38%)
  مكافآت     █████ 96 (16%)
  استعجال    ███████████ 213 (36%)
  حث         ██ 47 (8%)


## تحليل الأطوال


In [4]:
for label, name, data in [(0,'Phishing',phish),(1,'Safe',safe)]:
    lengths = data.str.len()
    print(f'{name:10s}: min={lengths.min():.0f}  median={lengths.median():.0f}  mean={lengths.mean():.0f}  max={lengths.max():.0f}')


Phishing  : min=61  median=103  mean=102  max=146
Safe      : min=43  median=84  mean=85  max=131


## ✅ الأنماط المكتشفة جاهزة لبناء الـ Generator

---
**التالي:** [03 - Generator Training](03_generator_training.ipynb)
